# 第十章：非线性降维与生成模型

## 10.1 t-SNE：可视化高维数据的局部结构

### PCA 的局限

PCA 只捕捉线性结构——它找的是方差最大的直线方向。但如果数据分布在弯曲的流形上（如瑞士卷），PCA 无法将其展开。t-SNE 的优化目标不同：只关心局部邻域——高维中接近的点在低维中也应该靠近；高维中远离的点在低维中放哪无所谓。

### t-SNE 的两步

**第一步**：高维空间中用高斯核定义相似性。对每个点 $x_i$，$x_j$ 与 $x_i$ 的相似性等于 $x_j$ 在以 $x_i$ 为中心的高斯分布下的概率密度。高斯核的带宽 $\sigma_i$ 通过困惑度参数自动调整。

**第二步**：低维空间中使用 Student $t$ 分布（自由度 1，概率密度 $f(x)=1/[\pi(1+x^2)]$，尾部比高斯重）。$t$ 分布的重尾使得中距离点在低维中被适度分开——解决了高维中"中等距离点对太多、低维空间放不下"的拥挤问题。

**优化**：最小化高维和低维相似性之间的 KL 散度，梯度下降直接优化低维坐标。数学细节见 10.1.1 节。



### 三种降维的本质区别

| 方法 | 优化目标 | 保留什么 | 复杂度 |
|------|---------|---------|--------|
| PCA (Principal Component Analysis) | $\max \text{Var}$ | 全局线性方差 | $\mathcal{O}(nd^2)$ |
| t-SNE | $\min \; \mathrm{KL}(P \| Q)$ | 局部邻域结构 | $\mathcal{O}(n^2)$ |
| UMAP | 模糊单纯形交叉熵 | 局部 + 部分全局 | $\mathcal{O}(n^{1.14})$ |

PCA 在高维空间中找方差最大的线性方向，将所有点沿这些方向投影——全局结构保留，但局部聚集关系丢失。t-SNE 反之：只看每个点的最近几十个邻居，尽力让邻居在低维中仍然靠近，远距离点对之间没有约束——局部精确，全局距离无意义。UMAP 折中：保留局部的同时尽量保持一定的全局结构，且计算速度远快于 t-SNE。



### 10.1.1 t-SNE 的数学原理

t-SNE 的核心思想：在高维空间中相似的样本点，在低维空间中也应该靠近。

#### Step 1: 高维相似性（高斯核）

对每个样本 $x_i$，定义它与其他样本的条件概率（"$x_j$ 是 $x_i$ 的邻居"）：

$$

p_{j|i} = \frac{\exp(-\|x_i - x_j\|^2 / 2\sigma_i^2)}{\sum_{k \neq i} \exp(-\|x_i - x_k\|^2 / 2\sigma_i^2)}

$$

$\sigma_i$ 由 **perplexity** 参数控制，使得每个样本的有效邻居数大致等于 perplexity。

对称化：$p_{ij} = \frac{p_{j|i} + p_{i|j}}{2n}$

#### Step 2: 低维嵌入（t 分布）

在低维空间中，用自由度为 1 的 Student t 分布（即 Cauchy 分布）来度量相似性：

$$

q_{ij} = \frac{(1 + \|y_i - y_j\|^2)^{-1}}{\sum_{k \neq l} (1 + \|y_k - y_l\|^2)^{-1}}

$$

#### Step 3: KL (Kullback-Leibler Divergence) 散度最小化

**KL 散度定义**（离散形式）：

$$D_{KL}(P \| Q) = \sum_i P(i) \cdot \log \frac{P(i)}{Q(i)}$$

度量用分布 $Q$ 近似分布 $P$ 时损失的信息量。当 $P(i)$ 很大但 $Q(i)$ 很小时，$\log(P/Q)$ 很大，惩罚极重——"真实分布中常见的事件，你的模型认为几乎不可能"。反向不对称：$P(i)$ 很小但 $Q(i)$ 很大时惩罚较轻。因此 $D_{KL}(P\|Q) \neq D_{KL}(Q\|P)$。

t-SNE 最小化 $P$ 和 $Q$ 之间的 KL 散度：

$$

\mathcal{L} = \sum_i \sum_j p_{ij} \log \frac{p_{ij}}{q_{ij}}

$$

#### 为什么用 t 分布而非高斯？

高斯分布的尾部太薄——中距离的点对在低维空间中要么挤在一起（无法区分），要么推得太远。t 分布的**重尾 (heavy tail)** 属性允许中距离点对在低维空间中适度分开，产生更好的聚类分离效果。

### 10.1.2 t-SNE vs UMAP：非线性降维的选择困境

| 维度 | t-SNE | UMAP |
|------|-------|------|
| 数学基础 | 概率（KL 散度） | 拓扑（模糊单纯形） |
| 全局结构 | 较差（距离无意义） | 较好（保留部分全局关系） |
| 速度 | $O(n^2)$ | $O(n^{1.14})$ |
| 可复现性 | 随机，不同运行结果不同 | 使用随机种子可复现 |
| 最佳场景 | 聚类可视化 | 大规模数据探索 |


In [ ]:
# === PCA vs t-SNE 对比 ===
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, make_blobs
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import time

# 生成复杂数据
np.random.seed(42)  # 固定随机种子，保证结果可复现
n_samples = 500

# 三维瑞士卷数据（Swiss Roll）
t = np.linspace(0, 15, n_samples)  # 生成等间隔序列
X_swiss = np.zeros((n_samples, 3))  # 创建全零数组
X_swiss[:, 0] = t * np.cos(t)
X_swiss[:, 1] = t * np.sin(t)
X_swiss[:, 2] = t
color = np.arctan2(X_swiss[:, 1], X_swiss[:, 0])

print("数据形状:", X_swiss.shape)

# PCA
t0 = time.time()
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_swiss)  # 拟合并应用变换
t_pca = time.time() - t0
print(f"PCA: {X_pca.shape}, 用时 {t_pca:.3f}s")

# t-SNE (需要 scikit-learn)
t0 = time.time()
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca')
X_tsne = tsne.fit_transform(X_swiss)  # 拟合并应用变换
t_tsne = time.time() - t0
print(f"t-SNE: {X_tsne.shape}, 用时 {t_tsne:.3f}s")

# 可视化对比
fig = plt.figure(figsize=(14, 5))
ax1 = fig.add_subplot(1, 3, 1, projection='3d')  # 3D
ax2 = fig.add_subplot(1, 3, 2)
ax3 = fig.add_subplot(1, 3, 3)
axes = [ax1, ax2, ax3]
# 创建三个子图（3D + 2D + 2D）
fig = plt.figure(figsize=(14, 5))
ax1 = fig.add_subplot(1, 3, 1, projection='3d')  # 3D：瑞士卷
ax2 = fig.add_subplot(1, 3, 2)                   # 2D：PCA
ax3 = fig.add_subplot(1, 3, 3)                   # 2D：t-SNE

# 3D 瑞士卷
ax1.scatter(X_swiss[:, 0], X_swiss[:, 1], X_swiss[:, 2],
            c=color, s=3, alpha=0.5)
ax1.set_title('Original 3D Data (Swiss Roll)')

# PCA 降维
ax2.scatter(X_pca[:, 0], X_pca[:, 1], c=color, cmap='viridis', s=5)
ax2.set_title('PCA (2D Projection)')

# t-SNE 降维
ax3.scatter(X_tsne[:, 0], X_tsne[:, 1], c=color, cmap='viridis', s=5)
ax3.set_title('t-SNE (2D Embedding)')

plt.tight_layout()
import os
os.makedirs('../fig', exist_ok=True)
plt.savefig('../fig/pca_tsne_comparison.png', dpi=100, bbox_inches='tight')
plt.show()  # 显示图表

print("\n关键观察:")
print(f"  PCA: 找到 {X_pca.shape[1]} 维 → 丢失了非线性结构")
print(f"  t-SNE: 找到 {X_tsne.shape[1]} 维 → 发现了 {len(np.unique(color))} 个聚类")


## 10.2 UMAP：更快更稳定的替代方案

**UMAP (Uniform Manifold Approximation and Projection, 2018)** 是比 t-SNE 更新的
非线性降维方法。它在保持全局结构方面优于 t-SNE，同时运行速度更快、结果可复现。

### t-SNE 的痛点

- **随机性**：不同运行给出不同结果
- **perplexity**：关键超参数，控制局部 vs 全局结构的平衡（典型值 5-50）
- **计算成本**：$O(n^2)$ 或更高

### 选择建议

- 需要**可复现**的结果：使用 UMAP
- 需要**探索性分析**：t-SNE 多次运行观察聚类稳定性
- 需要**速度**：PCA (Principal Component Analysis) 用于快速预览


In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits
# === 手写数字 t-SNE 可视化 ===
import matplotlib.pyplot as plt
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print(f"Digits 数据: {X_digits.shape}")

# PCA 快速预览
pca_digits = PCA(n_components=2).fit_transform(X_digits)  # 拟合并应用变换
print(f"PCA (2D): 前两个主成分")

# t-SNE
tsne_digits = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne_d = tsne_digits.fit_transform(X_digits)  # 拟合并应用变换
print(f"t-SNE (2D)")

# UMAP (如果可用)
try:
    import umap
    umap_reducer = umap.UMAP(n_components=2, random_state=42)
    X_umap = umap_reducer.fit_transform(X_digits)  # 拟合并应用变换
    print(f"UMAP (2D): {X_umap.shape}")
except ImportError:
    print("UMAP 未安装。安装: pip install umap-learn")

# 可视化
fig, axes = plt.subplots(1, 4, figsize=(16, 6))  # 创建子图网格
plot_data = [
    ('PCA', pca_digits),
    ('t-SNE', X_tsne_d),
]
try:
    plot_data.append(('UMAP', X_umap))
except NameError:
    pass

for i, (title, data) in enumerate(plot_data):
    ax = axes[i]
    scatter = ax.scatter(data[:, 0], data[:, 1], c=y_digits, cmap='tab10', s=5, alpha=0.6)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])

axes[0].legend(fontsize=8)
plt.tight_layout()  # 自动调整子图间距
plt.savefig('../fig/tsne_umap_digits.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
plt.show()  # 显示图表


## 10.3 自编码器 (Autoencoder)：生成模型的基石

自编码器是最简单的**生成模型**——它学习将输入复制到输出。
虽然看起来"什么都没做"，但这个"复制"任务迫使网络学习
数据的**紧凑表示 (compact representation)**。

### 自编码器架构

```
输入 x → [Encoder] → 潜在码 z → [Decoder] → 重建 x̂
```

### 数学定义

- **Encoder**: $\mathbf{z} = f_\theta(\mathbf{x})$ 将输入压缩为低维潜在表示
- **Decoder**: $\hat{\mathbf{x}} = g_\phi(\mathbf{z})$ 从潜在码重建输入
- **损失函数**: $\mathcal{L} = \|\mathbf{x} - \hat{\mathbf{x}}\|^2$

### 为什么需要自编码器？

1. **降维**：学到数据的紧凑表示 → 可用于可视化、压缩、去噪
2. **生成**：从潜在空间采样 → 可生成新数据
3. **异常检测**：重建误差大的样本 → 可能是异常值


### 10.3.1 潜在空间与流形假设

自编码器的核心产物是**潜在空间 (Latent Space)**——将高维数据压缩到的低维连续空间。

#### 流形假设 (Manifold Hypothesis)

真实世界的高维数据（如图像、声音）虽然表面上是高维的（一张 28×28 图片 = 784 维），但实际上分布在低维流形上——所有合法的 MNIST 手写数字只占据 784 维空间中的一个极小区域。

自编码器做的事情就是**学习这个流形的参数化**：Encoder 将流形上的点映射到潜在空间坐标，Decoder 将坐标映射回流形上的点。

#### 为什么潜在空间是连续的？

一个训练良好的自编码器的潜在空间具有以下性质：
- **连续性**：潜在空间中相邻的点解码后是相似的图像
- **可插值**：在潜在空间中两个点之间线性插值，解码后得到平滑过渡的图像

这就是为什么 VAE (Variational Autoencoder) 可以从潜在空间随机采样生成新图像——它学会了整个流形，而不仅仅是训练样本点。


In [ ]:
# === 简单自编码器 (MNIST) ===
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 自动选择设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 自编码器：784 -> 128 -> 64 -> 32 -> 64 -> 128 -> 784
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 784),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten (batch,1,28,28) -> (batch,784)
        z = self.encoder(x)        # (batch,784) -> (batch,32)
        x_hat = self.decoder(z)    # (batch,32) -> (batch,784)
        return x_hat

# 加载 MNIST
transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root='../data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

model = Autoencoder().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(f"模型参数: {sum(p.numel() for p in model.parameters()):,}")

# 训练（3 epochs 快速演示）
model.train()
for epoch in range(3):
    total_loss = 0
    for x_batch, _ in train_loader:
        x_batch = x_batch.to(device)
        optimizer.zero_grad()
        x_hat = model(x_batch)
        loss = criterion(x_hat, x_batch.view(x_batch.size(0), -1).to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/3, Loss: {total_loss/len(train_loader):.4f}")

# 可视化：原图 vs 重建 vs 随机生成
model.eval()
sample, _ = train_dataset[0]
sample = sample.unsqueeze(0).to(device)
with torch.no_grad():
    recon = model(sample).cpu().view(28, 28)

# 随机生成：从标准正态采样潜在向量通过解码器
z_random = torch.randn(1, 32).to(device)
with torch.no_grad():
    generated = model.decoder(z_random).cpu().view(28, 28)

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(train_dataset[0][0].view(28, 28), cmap='gray')
axes[0].set_title('Original')
axes[1].imshow(recon, cmap='gray')
axes[1].set_title('Reconstructed')
axes[2].imshow(generated, cmap='gray')
axes[2].set_title('Random Generation')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.savefig('../fig/autoencoder_demo.png', dpi=100, bbox_inches='tight')
plt.show()


## 10.4 VAE：生成新样本的编码器-解码器

### 自编码器的局限

自编码器把输入压缩到低维潜在空间再重建出来。它可以做降维和去噪，但**不能生成新样本**——因为潜在空间是"散点"：每个训练样本对应一个点，点与点之间的区域没有意义，随机取一个潜在点解码出来可能是噪声。

VAE 解决这个问题的方式：不直接把输入映射为一个点，而是映射为一个**概率分布**（均值和方差）。这样潜在空间变成连续的——相近的潜在点解码出相似的输出，可以在空间中平滑插值。

### Reparameterization Trick

编码器输出 $\mu$ 和 $\sigma$（正态分布的参数），但"从 $\mathcal{N}(\mu, \sigma^2)$ 采样"这个操作不可导——无法反向传播。VAE 的解决方案：从标准正态 $\mathcal{N}(0, 1)$ 采样 $\epsilon$，然后计算 $z = \mu + \sigma \cdot \epsilon$。$z$ 对 $\mu$ 和 $\sigma$ 可导，梯度可以通过 $\epsilon$ 传回编码器。

### 损失函数的两个部分

$$\mathcal{L} = \underbrace{\mathbb{E}_{z \sim q(z|x)}[\log p(x|z)]}_{\text{重建损失}} - \underbrace{D_{KL}(q(z|x) \| p(z))}_{\text{KL 正则}}$$

**重建损失：** 从潜在变量 $z$ 重建出 $x$，和自编码器一样——让输出尽可能接近输入。

**KL 正则：** $D_{KL}(q(z|x) \| p(z))$ 强制编码器输出的分布 $q(z|x)$ 接近标准正态 $p(z) = \mathcal{N}(0, I)$。如果 $D_{KL}=0$，潜在变量 $z$ 完全服从标准正态——可以从 $\mathcal{N}(0, I)$ 随机采样任意 $z$，解码器都能生成有意义的样本。如果不加这个约束，编码器会把每个输入的方差压到 0（退化回普通自编码器——每个输入映射为一个确定的潜在点，潜在空间中大部分区域无意义，失去生成能力）。KL 项强迫每个输入对应一个分布而非一个点，保证潜在空间连续可采样——这是 VAE 能生成新样本的关键。


In [ ]:
# === VAE 实现 (MNIST) ===
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class VAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.latent_dim = latent_dim
        # Encoder: 784 -> 256 -> 128 -> (mu, logvar) each of size latent_dim
        self.encoder = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        # Decoder: latent_dim -> 128 -> 256 -> 784
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 784),
            nn.Sigmoid()
        )

    def encode(self, x):
        x = x.view(x.size(0), -1)          # flatten (batch,1,28,28) -> (batch,784)
        h = self.encoder(x)                # (batch,784) -> (batch,128)
        return self.fc_mu(h), self.fc_logvar(h)  # each (batch, latent_dim)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)             # (batch, latent_dim) -> (batch, 784)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

# 加载 MNIST
transform = transforms.ToTensor()
dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
loader = DataLoader(dataset, batch_size=128, shuffle=True)

model = VAE(latent_dim=2).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 训练（5 epochs 快速演示）
model.train()
for epoch in range(5):
    total_loss = 0
    for x_batch, _ in loader:
        x_batch = x_batch.to(device)
        optimizer.zero_grad()
        recon, mu, logvar = model(x_batch)
        # 重建损失：MSE between recon (batch,784) and flattened x_batch (batch,784)
        loss_recon = F.mse_loss(recon, x_batch.view(x_batch.size(0), -1), reduction='sum') / x_batch.size(0)
        # KL 散度：-0.5 * sum(1 + logvar - mu^2 - exp(logvar))
        loss_kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x_batch.size(0)
        loss = loss_recon + 0.1 * loss_kl
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={total_loss/len(loader):.4f}  recon={loss_recon.item():.4f}  kl={loss_kl.item():.4f}")

# 生成与重建
model.eval()
with torch.no_grad():
    # 随机生成：从 N(0,1) 采样 -> 解码
    z = torch.randn(16, 2).to(device)
    generated = model.decode(z).cpu().view(-1, 1, 28, 28)
    
    # 重建：原图 -> 编码 -> 解码
    sample, _ = dataset[0]
    sample = sample.unsqueeze(0).to(device)
    recon, _, _ = model(sample)
    recon = recon.cpu().view(28, 28)

fig, axes = plt.subplots(2, 8, figsize=(12, 4))
axes[0, 0].imshow(dataset[0][0].view(28, 28), cmap='gray')
axes[0, 0].set_title('Original')
axes[0, 1].imshow(recon, cmap='gray')
axes[0, 1].set_title('Reconstructed')
for i in range(8):
    axes[1, i].imshow(generated[i, 0], cmap='gray')
    axes[1, i].set_title(f'Gen {i+1}')
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.savefig('../fig/vae_demo.png', dpi=100, bbox_inches='tight')
plt.show()


## 10.5 GAN 基础回顾

### 生成器和判别器各自是什么

**生成器 $G$** 是一个神经网络，输入是随机噪声向量 $\mathbf{z}$（通常从标准正态分布采样，维度如 100），输出是一张假图片。$G$ 内部通常用转置卷积（或全连接层）将低维噪声逐步上采样到目标图片尺寸。不同 $\mathbf{z}$ 产生不同的图片——$\mathbf{z}$ 的每个维度控制生成图片的某个抽象属性。生成器没有任何"看到真实图片"的路径，它唯一的信息来源是判别器传回的梯度。

**判别器 $D$** 也是一个神经网络，输入是一张图片（真的或假的），输出是一个标量（0 到 1 之间），表示"这张图是真实样本的概率"。$D$ 内部通常是卷积网络，逐步下采样提取特征，最后通过全连接层输出一个值。$D$ 训练时能看到真实图片和 $G$ 生成的假图片，学会区分两者。

### 两者如何互动：对抗训练

$G$ 和 $D$ 交替训练，目标相反。$D$ 想准确分辨真伪——真实图片输出 1，生成图片输出 0。$G$ 想让 $D$ 把自己的假图片判为 1。两者互相推动：$D$ 变强迫使 $G$ 生成更逼真的图片以骗过 $D$，$G$ 变强迫使 $D$ 学习更精细的辨别能力。

### 损失函数与极小极大博弈

$G$ 和 $D$ 的目标相反，构成一个极小极大优化问题：

$$\min_G \max_D \; V(D, G) = \mathbb{E}_{\mathbf{x} \sim p_{\text{data}}}[\log D(\mathbf{x})] + \mathbb{E}_{\mathbf{z} \sim p_{\mathbf{z}}}[\log(1 - D(G(\mathbf{z})))]$$

**判别器 $D$ 的视角：** $D(\mathbf{x})$ 输出判断 $\mathbf{x}$ 为真实样本的概率。$D$ 希望最大化整个式子——让 $\log D(\mathbf{x})$ 大（对真实样本输出高概率），让 $\log(1 - D(G(\mathbf{z})))$ 大（对生成样本输出低概率，使 $1-D(G(\mathbf{z}))$ 接近 1）。

**生成器 $G$ 的视角：** $G$ 只想最小化第二项——让 $D(G(\mathbf{z}))$ 接近 1，即使得 $\log(1 - D(G(\mathbf{z})))$ 趋近于 $-\infty$。$G$ 不控制第一项（与真实样本无关）。

### 最优判别器的推导

固定 $G$，对 $D$ 求最优解。将期望展开为积分：

$$V(D, G) = \int_{\mathbf{x}} \left[ p_{\text{data}}(\mathbf{x}) \log D(\mathbf{x}) + p_g(\mathbf{x}) \log(1 - D(\mathbf{x})) \right] d\mathbf{x}$$

其中 $p_g$ 是生成器产生的样本分布。对 $D(\mathbf{x})$ 求导并令为零：

$$\frac{\partial}{\partial D(\mathbf{x})} \left[ p_{\text{data}} \log D + p_g \log(1-D) \right] = \frac{p_{\text{data}}}{D} - \frac{p_g}{1-D} = 0$$

$$\Rightarrow \frac{p_{\text{data}}}{D} = \frac{p_g}{1-D} \quad \Rightarrow \quad D^*(\mathbf{x}) = \frac{p_{\text{data}}(\mathbf{x})}{p_{\text{data}}(\mathbf{x}) + p_g(\mathbf{x})}$$

当 $p_{\text{data}} = p_g$（生成分布完美匹配真实分布）时，$D^* = 0.5$——判别器无法区分真伪，最优分类准确率只有 50%。

### 最优判别器下的生成器目标

将 $D^*$ 代入 $V(D,G)$：

$$V(D^*, G) = \mathbb{E}_{\mathbf{x} \sim p_{\text{data}}}\left[\log \frac{p_{\text{data}}}{p_{\text{data}} + p_g}\right] + \mathbb{E}_{\mathbf{x} \sim p_g}\left[\log \frac{p_g}{p_{\text{data}} + p_g}\right]$$

整理后等价于：

$$V(D^*, G) = 2 \cdot D_{JS}(p_{\text{data}} \| p_g) - 2\log 2$$

即生成器在最优判别器下的目标等价于最小化 JS 散度。JS 散度度量两个分布的距离——当 $p_{\text{data}} = p_g$ 时 $D_{JS} = 0$，当两个分布完全不重叠时 $D_{JS} = \log 2$（常数）。

### 训练中的梯度消失问题

当 $D$ 接近 $D^*$ 时，$G$ 的梯度正比于 JS 散度的梯度。但如果 $p_{\text{data}}$ 和 $p_g$ 的支撑集不重叠（训练初期生成图片完全不像真实图片，两种分布几乎无交集），JS 散度退化为常数 $\log 2$，梯度为零——$G$ 收不到任何信号，训练停滞。这就是 GAN 训练不稳定的数学根源。


## 10.6 DDcGAN(Dual-Discriminator Conditional Generative Adversarial Network)：双判别器条件对抗网络

### 从 GAN 到条件 GAN

原始 GAN 只能"随机生成"——给一个噪声向量 $\mathbf{z}$，生成一张图片，但无法控制生成什么。**条件 GAN (Conditional GAN)** 引入条件信息 $c$（如类别标签）：

$$\text{生成器：} \quad G(\mathbf{z}, c) \rightarrow \hat{\mathbf{x}} \quad \text{（指定类别 c，生成该类别的样本）}$$

$$\text{判别器：} \quad D(\mathbf{x}, c) \rightarrow [0, 1] \quad \text{（判断 x 是否是类别 c 的真实样本）}$$

判别器的任务变了——不仅判断"真不真"，还要判断"是不是这个类别的"。一张真实的猫图片配上"狗"的标签，判别器应该判 fake。这迫使生成器不仅生成逼真图片，还要生成**与条件匹配**的图片。

### 单判别器的根本矛盾

判别器需要同时评估两种质量：

1. **全局合理性**：整体结构对不对——猫有没有四条腿、脸的比例是否协调。需要**大感受野**——看到整张图才能判断整体结构
2. **局部真实性**：细节纹理好不好——毛发是否清晰、边缘是否锐利。需要**小感受野**——聚焦局部块才能捕捉高频细节

这两个需求在网络结构上直接冲突。大感受野需要深层网络+大量下采样（丢失细节信息），小感受野需要浅层网络+保留高分辨率（看不到全局）。单个判别器只能折中——结果是全局和局部都不够好。实践中表现为：整体结构对但细节模糊，或细节清晰但结构扭曲。

### DDcGAN 的解法：分而治之

**DDcGAN (Dual-Discriminator Conditional GAN)** 用两个结构不同的判别器分别处理两种质量：

```
                       ┌─→ 全局判别器 D_g（深层网络，看整张图）→ 整体结构分数
生成器 G(z, c) → 生成图 ─┤
                       └─→ 局部判别器 D_l（浅层网络，看随机裁剪块）→ 细节质量分数
```

**全局判别器 $D_g$**：接收完整图片（如 64×64），通过多层卷积逐步下采样到 4×4 再打分。深层结构给它足够大的感受野覆盖整张图——它对"猫头接在狗身上"这类结构错误敏感，但对模糊的毛发不敏感（下采样丢失了高频信息）。

**局部判别器 $D_l$**：接收从图中**随机裁剪**的小块（如 16×16），浅层网络直接处理。它看不到全局结构（一块 16×16 的区域不知道自己是猫耳朵还是狗尾巴），但对纹理质量极其敏感——模糊、噪点、棋盘伪影都逃不过。

随机裁剪是关键设计：每次训练迭代裁不同的位置，局部判别器等效于在**所有可能的局部块**上评估细节质量，而不是固定看某个区域。

### 两个判别器如何协同训练生成器

生成器同时接收两个判别器的梯度信号：

- $D_g$ 的梯度告诉生成器"整体结构往哪个方向改"
- $D_l$ 的梯度告诉生成器"局部纹理往哪个方向改"

两个信号叠加（按权重 $\lambda_{\text{local}}$ 组合），生成器在一次参数更新中同时改善结构和细节。这就是"分而治之"的收益：两个专职判别器给出的梯度信号，比一个全能判别器的折中信号更精确。


## 10.7 DDcGAN 损失函数详解

DDcGAN 使用两个结构不同的判别器——**全局判别器 $D_g$** 和**局部判别器 $D_l$**——分别从不同尺度评估生成质量。

### 对抗损失：Hinge Loss

DDcGAN 使用 Hinge Loss 替代原始 GAN 的交叉熵：

**判别器损失：**

$$\mathcal{L}_{D} = \mathbb{E}_{\mathbf{x} \sim p_{\text{data}}}[\max(0, 1 - D(\mathbf{x}))] + \mathbb{E}_{\mathbf{z} \sim p_{\mathbf{z}}}[\max(0, 1 + D(G(\mathbf{z})))]$$

Hinge Loss 的直观：$D(\mathbf{x})$ 只需要超过 1（对真实样本）或低于 -1（对生成样本）就算"足够好"，不再继续优化。这避免了判别器过于自信导致梯度消失——即使 $D$ 很强大，只要它在 ±1 范围内，梯度仍然存在。

**生成器损失：**

$$\mathcal{L}_{G} = -\mathbb{E}_{\mathbf{z} \sim p_{\mathbf{z}}}[D(G(\mathbf{z}))]$$

生成器只需要最大化判别器对生成样本的打分——让 $D(G(\mathbf{z}))$ 尽可能大。

### 双判别器联合损失

DDcGAN 的核心创新：两个判别器独立优化，但共享生成器。全局判别器看整张图片的合理性，局部判别器看随机裁切区域的细节质量：

$$\mathcal{L}_{D}^{\text{total}} = \mathcal{L}_{D_g} + \lambda_{\text{local}} \cdot \mathcal{L}_{D_l}$$

### 为什么双判别器比单判别器更稳定

单判别器需要在全局语义和局部细节之间做权衡——两者需要的网络结构不同（大感受野 vs 小感受野），导致训练在两个目标之间摇摆。双判别器将这个多目标问题分解为两个独立的单目标问题——每个判别器专注一个尺度，不需要内部折中。

条件信息通过**投影判别器**注入：$D(\mathbf{x}, c) = \mathbf{w}_c^T \phi(\mathbf{x}) + \psi(\phi(\mathbf{x}))$，其中 $\phi(\mathbf{x})$ 是特征提取器，$\mathbf{w}_c$ 是类别 $c$ 对应的投影向量。投影的内积 $\mathbf{w}_c^T \phi(\mathbf{x})$ 直接度量"特征 $\phi(\mathbf{x})$ 与类别 $c$ 的匹配程度"。这与简单拼接条件向量的方式不同——拼接让网络自行学习条件与特征的关系，投影则显式建模为内积，提供了更强的归纳偏置。


In [ ]:
# === DDcGAN 核心组件实现（PyTorch）===
import torch
import torch.nn as nn
import torch.nn.functional as F

class Generator(nn.Module):  # 所有网络层的基类
    """DDcGAN 生成器：噪声 + 条件 → 图像
    使用转置卷积 (ConvTranspose2d) 从低分辨率逐步上采样
    """
    def __init__(self, latent_dim=100, num_classes=10, img_channels=3, feature_dim=64):
        super().__init__()
        self.latent_dim = latent_dim
        self.num_classes = num_classes
        
        # 条件嵌入：将类别标签映射到 latent 空间
        self.label_embedding = nn.Embedding(num_classes, latent_dim)  # 词嵌入层
        
        # 生成器主体：latent_dim*2 (noise + embed) → 4×4×512 → ... → 32×32×3
        self.main = nn.Sequential(  # 顺序容器
            # 输入: (latent_dim*2) × 1 × 1
            nn.ConvTranspose2d(latent_dim * 2, feature_dim * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(feature_dim * 8),  # 二维批归一化
            nn.ReLU(True),
            # 状态: (512) × 4 × 4
            
            nn.ConvTranspose2d(feature_dim * 8, feature_dim * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_dim * 4),  # 二维批归一化
            nn.ReLU(True),
            # 状态: (256) × 8 × 8
            
            nn.ConvTranspose2d(feature_dim * 4, feature_dim * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_dim * 2),  # 二维批归一化
            nn.ReLU(True),
            # 状态: (128) × 16 × 16
            
            nn.ConvTranspose2d(feature_dim * 2, img_channels, 4, 2, 1, bias=False),
            nn.Tanh()  # 输出范围 [-1, 1]
            # 状态: (3) × 32 × 32
        )
    
    def forward(self, z, labels):
        # 条件嵌入
        c = self.label_embedding(labels).unsqueeze(-1).unsqueeze(-1)  # 增加一个维度
        # 拼接噪声和条件
        z = z.unsqueeze(-1).unsqueeze(-1)  # 增加一个维度
        z_c = torch.cat([z, c], dim=1)  # (batch, latent_dim*2, 1, 1)
        return self.main(z_c)  # 返回结果

# ========================================
class GlobalDiscriminator(nn.Module):  # 所有网络层的基类
    """全局判别器：关注整体图像结构
    使用较深的网络 + 大感受野
    """
    def __init__(self, num_classes=10, img_channels=3, feature_dim=64):
        super().__init__()
        
        # 条件嵌入（投影判别器风格）
        self.label_embedding = nn.Embedding(num_classes, feature_dim * 8)  # 词嵌入层
        
        self.main = nn.Sequential(  # 顺序容器
            # 输入: 3 × 32 × 32
            nn.Conv2d(img_channels, feature_dim, 4, 2, 1, bias=False),  # 二维卷积层
            nn.LeakyReLU(0.2, inplace=True),
            # (64) × 16 × 16
            
            nn.Conv2d(feature_dim, feature_dim * 2, 4, 2, 1, bias=False),  # 二维卷积层
            nn.BatchNorm2d(feature_dim * 2),  # 二维批归一化
            nn.LeakyReLU(0.2, inplace=True),
            # (128) × 8 × 8
            
            nn.Conv2d(feature_dim * 2, feature_dim * 4, 4, 2, 1, bias=False),  # 二维卷积层
            nn.BatchNorm2d(feature_dim * 4),  # 二维批归一化
            nn.LeakyReLU(0.2, inplace=True),
            # (256) × 4 × 4
            
            nn.Conv2d(feature_dim * 4, feature_dim * 8, 4, 1, 0, bias=False),  # 二维卷积层
            nn.BatchNorm2d(feature_dim * 8),  # 二维批归一化
            nn.LeakyReLU(0.2, inplace=True),
            # (512) × 1 × 1
        )
    
    def forward(self, img, labels):
        features = self.main(img)  # (batch, 512, 1, 1)
        features = features.view(features.size(0), -1)  # (batch, 512)
        
        # 投影判别器：内积形式注入条件
        label_embed = self.label_embedding(labels)  # (batch, 512)
        output = (features * label_embed).sum(dim=1, keepdim=True)  # (batch, 1)
        return output  # 返回结果

# ========================================
class LocalDiscriminator(nn.Module):  # 所有网络层的基类
    """局部判别器：关注随机图像块的纹理和细节
    使用较浅的网络 + 局部感受野
    """
    def __init__(self, num_classes=10, img_channels=3, feature_dim=32):
        super().__init__()
        
        self.label_embedding = nn.Embedding(num_classes, feature_dim * 4)  # 词嵌入层
        
        self.main = nn.Sequential(  # 顺序容器
            # 输入: 3 × 16 × 16 (从原图随机裁剪)
            nn.Conv2d(img_channels, feature_dim, 4, 2, 1, bias=False),  # 二维卷积层
            nn.LeakyReLU(0.2, inplace=True),
            # (32) × 8 × 8
            
            nn.Conv2d(feature_dim, feature_dim * 2, 4, 2, 1, bias=False),  # 二维卷积层
            nn.BatchNorm2d(feature_dim * 2),  # 二维批归一化
            nn.LeakyReLU(0.2, inplace=True),
            # (64) × 4 × 4
            
            nn.Conv2d(feature_dim * 2, feature_dim * 4, 4, 1, 0, bias=False),  # 二维卷积层
            nn.BatchNorm2d(feature_dim * 4),  # 二维批归一化
            nn.LeakyReLU(0.2, inplace=True),
            # (128) × 1 × 1
        )
    
    def forward(self, img_patch, labels):
        features = self.main(img_patch)
        features = features.view(features.size(0), -1)  # 改变张量形状（不拷贝数据）
        label_embed = self.label_embedding(labels)
        output = (features * label_embed).sum(dim=1, keepdim=True)  # 沿指定维度求和
        return output  # 返回结果

# --- 损失函数 ---
# === DDcGAN 损失函数 ===
import torch.nn.functional as F
class DDcGANLoss:
    """DDcGAN Hinge Loss + 双判别器加权"""
    
    def __init__(self, lambda_local=0.5):
        self.lambda_local = lambda_local
    
    def discriminator_loss(self, D_global_out_real, D_global_out_fake,
                            D_local_out_real, D_local_out_fake):
        """
        Hinge Loss for both discriminators
        D_real 应 >= 1, D_fake 应 <= -1
        """
        # 全局判别器
        loss_D_global_real = F.relu(1.0 - D_global_out_real).mean()  # 沿指定维度求均值
        loss_D_global_fake = F.relu(1.0 + D_global_out_fake).mean()  # 沿指定维度求均值
        loss_D_global = loss_D_global_real + loss_D_global_fake
        
        # 局部判别器
        loss_D_local_real = F.relu(1.0 - D_local_out_real).mean()  # 沿指定维度求均值
        loss_D_local_fake = F.relu(1.0 + D_local_out_fake).mean()  # 沿指定维度求均值
        loss_D_local = loss_D_local_real + loss_D_local_fake
        
        return loss_D_global + self.lambda_local * loss_D_local  # 返回结果
    
    def generator_loss(self, D_global_out_fake, D_local_out_fake):
        """生成器希望 D(G(z)) 尽可能大"""
        loss_G_global = -D_global_out_fake.mean()  # 沿指定维度求均值
        loss_G_local = -D_local_out_fake.mean()  # 沿指定维度求均值
        return loss_G_global + self.lambda_local * loss_G_local  # 返回结果


In [ ]:
# === DDcGAN 训练 Demo：在 MNIST 上生成手写数字 ===
# 使用简化版（灰度图 1 通道）训练少量 epoch 演示

import torch
import torch.optim as optim
def train_ddcgan_demo(epochs=10, batch_size=64, latent_dim=100, device='cpu'):
    """DDcGAN 在 MNIST 上的训练演示"""
    from torchvision import datasets, transforms
    
    # 数据加载
    transform = transforms.Compose([
        transforms.Resize(32),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])  # → [-1, 1]
    ])
    
    dataset = datasets.MNIST('./data', train=True,
                              download=True, transform=transform)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # 模型（灰度图 1 channel）
    G = Generator(latent_dim=latent_dim, num_classes=10, img_channels=1).to(device)  # 将数据移到 GPU 或 CPU
    D_global = GlobalDiscriminator(num_classes=10, img_channels=1).to(device)  # 将数据移到 GPU 或 CPU
    D_local = LocalDiscriminator(num_classes=10, img_channels=1).to(device)  # 将数据移到 GPU 或 CPU
    
    criterion = DDcGANLoss(lambda_local=0.5)
    
    # 优化器
    opt_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_D = optim.Adam(
        list(D_global.parameters()) + list(D_local.parameters()),
        lr=2e-4, betas=(0.5, 0.999)
    )
    
    history = {'G_loss': [], 'D_loss': []}
    
    for epoch in range(epochs):
        for i, (real_imgs, labels) in enumerate(dataloader):
            real_imgs, labels = real_imgs.to(device), labels.to(device)  # 将数据移到 GPU 或 CPU
            batch = real_imgs.size(0)
            
            # ---- 1. 训练判别器 ----
            opt_D.zero_grad()  # 清空上一轮的梯度
            
            # 生成假样本
            z = torch.randn(batch, latent_dim, device=device)  # 标准正态分布随机张量
            fake_imgs = G(z, labels).detach()  # detach: 不传播到 G
            
            # 获取局部 patch（随机裁剪 16×16）
            _, _, h, w = real_imgs.shape
            top = torch.randint(0, h - 16 + 1, (1,)).item() if h > 16 else 0  # 取出单个数值
            left = torch.randint(0, w - 16 + 1, (1,)).item() if w > 16 else 0  # 取出单个数值
            real_patches = real_imgs[:, :, top:top+16, left:left+16]
            fake_patches = fake_imgs[:, :, top:top+16, left:left+16]
            
            # 判别器输出
            Dg_real = D_global(real_imgs, labels)
            Dg_fake = D_global(fake_imgs, labels)
            Dl_real = D_local(real_patches, labels)
            Dl_fake = D_local(fake_patches, labels)
            
            loss_D = criterion.discriminator_loss(Dg_real, Dg_fake, Dl_real, Dl_fake)
            loss_D.backward()  # 反向传播，计算梯度
            opt_D.step()  # 用梯度更新参数
            
            # ---- 2. 训练生成器 ----
            opt_G.zero_grad()  # 清空上一轮的梯度
            
            z = torch.randn(batch, latent_dim, device=device)  # 标准正态分布随机张量
            fake_imgs = G(z, labels)
            
            # 局部 patch
            _, _, h, w = fake_imgs.shape
            top2 = torch.randint(0, max(h-16+1, 1), (1,)).item()  # 取出单个数值
            left2 = torch.randint(0, max(w-16+1, 1), (1,)).item()  # 取出单个数值
            fake_patches = fake_imgs[:, :, top2:top2+16, left2:left2+16]
            
            Dg_fake = D_global(fake_imgs, labels)
            Dl_fake = D_local(fake_patches, labels)
            
            loss_G = criterion.generator_loss(Dg_fake, Dl_fake)
            loss_G.backward()  # 反向传播，计算梯度
            opt_G.step()  # 用梯度更新参数
            
            if i % 150 == 0:
                history['D_loss'].append(loss_D.item())  # 取出单个数值
                history['G_loss'].append(loss_G.item())  # 取出单个数值
        
        print(f"Epoch {epoch+1:2d}/{epochs} | D Loss: {loss_D.item():.4f} | G Loss: {loss_G.item():.4f}")  # 取出单个数值
    
    return G, D_global, D_local, history  # 返回结果

# 运行训练（CPU 模式下较快完成）
device_train = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
G_trained, _, _, history = train_ddcgan_demo(epochs=5, batch_size=64, device=device_train)


In [ ]:
# === DDcGAN 生成结果可视化 ===
import matplotlib.pyplot as plt
import torch
@torch.no_grad()
def generate_and_show(G, num_classes=10, samples_per_class=8, device='cpu'):
    G.eval()  # 切换评估模式
    latent_dim = G.latent_dim
    
    fig, axes = plt.subplots(num_classes, samples_per_class, figsize=(samples_per_class, num_classes))  # 创建子图网格
    
    for cls in range(num_classes):
        z = torch.randn(samples_per_class, latent_dim, device=device)  # 标准正态分布随机张量
        labels = torch.full((samples_per_class,), cls, dtype=torch.long, device=device)
        fake_imgs = G(z, labels).cpu()  # 移到 CPU
        
        for j in range(samples_per_class):
            img = fake_imgs[j].squeeze().numpy()  # 去除大小为 1 的维度
            img = (img + 1) / 2  # [-1,1] → [0,1]
            axes[cls, j].imshow(img, cmap='gray')
            axes[cls, j].axis('off')
            if j == 0:
                axes[cls, j].set_ylabel(f'Class {cls}', fontsize=10, rotation=0, labelpad=20)
    
    plt.suptitle('DDcGAN Generated MNIST Digits (5 epochs)', fontsize=14, y=1.02)
    plt.tight_layout()  # 自动调整子图间距
    plt.savefig('../fig/ddcgan_generated.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
    plt.show()  # 显示图表

generate_and_show(G_trained, device=device_train)

# 训练曲线
if history['G_loss']:
    fig, ax = plt.subplots(figsize=(8, 4))  # 创建子图网格
    steps = range(len(history['G_loss']))
    ax.plot(steps, history['D_loss'], label='D Loss', alpha=0.8)
    ax.plot(steps, history['G_loss'], label='G Loss', alpha=0.8)
    ax.set_xlabel('Training Steps (×150)'); ax.set_ylabel('Loss')
    ax.set_title('DDcGAN Training Curves (Hinge Loss)')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()  # 自动调整子图间距
    plt.savefig('../fig/ddcgan_training.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
    plt.show()  # 显示图表


## 10.8 DDcGAN 关键设计要点

### 1. 投影判别器：条件信息的注入方式

将条件 $c$（类别标签）告诉判别器有多种方式，效果差异显著：

**方式一：拼接 (Concatenation)。** 把类别的 one-hot 向量拼到输入图片或中间特征上。问题：网络需要自己学会"这些拼接的维度代表类别，要和图片特征关联起来"——这个关联信号很弱，训练慢且不稳定。

**方式二：投影 (Projection)。** DDcGAN 采用的方式：

$$D(\mathbf{x}, c) = \underbrace{\mathbf{w}_c^T \phi(\mathbf{x})}_{\text{条件相关分数}} + \underbrace{\psi(\phi(\mathbf{x}))}_{\text{无条件分数}}$$

$\phi(\mathbf{x})$ 是卷积特征提取器的输出向量。$\mathbf{w}_c$ 是类别 $c$ 的嵌入向量（每个类别一个，可学习）。$\psi$ 是标量输出层。

投影的含义：$\mathbf{w}_c^T \phi(\mathbf{x})$ 是特征与类别嵌入的**内积**——直接度量"图片特征与类别 c 的匹配程度"。类别信息不再是拼接的附加维度，而是作为一个方向向量，特征在这个方向上的投影长度就是条件匹配分数。

内积结构是一个强归纳偏置：它显式假设"类别匹配度可以用特征空间中的方向对齐来衡量"。这个假设与 softmax 分类器的数学结构一致（logit = 权重向量与特征的内积），实践中训练更稳定、条件遵循度更高。

### 2. 局部判别器的随机裁剪策略

局部判别器每次迭代从生成图和真实图上随机裁剪相同位置的块。设计细节：

- **裁剪尺寸**：过大则退化为第二个全局判别器，过小则只能看到噪声级别的信息。经验值为全图边长的 1/4（64×64 图用 16×16 块）
- **位置随机**：均匀分布采样裁剪位置。长期来看每个空间位置被评估的概率相同，等效于全图密集的细节监督
- **真假对应**：真实图和生成图裁剪**相同位置**，保证判别器比较的是同位置的纹理质量而非位置差异

### 3. 架构不对称设计

两个判别器如果结构相同，会退化为集成 (ensemble)——两个相似的评委给出相似的意见，第二个判别器的边际价值趋近于零。DDcGAN 刻意让两者不对称：

| | 全局判别器 $D_g$ | 局部判别器 $D_l$ |
|---|---|---|
| 输入 | 完整图片 64×64 | 随机裁剪 16×16 |
| 深度 | 深（5-6 层卷积） | 浅（2-3 层卷积） |
| 感受野 | 覆盖全图 | 覆盖局部块 |
| 下采样 | 多次（64→4） | 少（16→4） |
| 敏感对象 | 结构错误、比例失调 | 模糊、噪点、伪影 |

不对称保证了两个判别器捕捉**互补**的失败模式——这是"双判别器"设计有效的前提。如果两个判别器高度相关，联合损失几乎等价于单判别器乘以 2，失去分而治之的意义。

### 4. 训练平衡：判别器不能太强

GAN 训练的经典问题在 DDcGAN 中被放大——现在有**两个**判别器压制生成器。若判别器过强（能完美分辨真伪），生成器的梯度消失，训练停滞。DDcGAN 的缓解手段：

- **Hinge Loss**：判别器分数超过 ±1 后不再产生梯度，防止判别器过度自信（见 10.7 节）
- **学习率不对称**：判别器学习率低于生成器（如 1:4），限制判别器的进步速度
- **谱归一化 (Spectral Normalization)**：约束判别器每层权重的最大奇异值，限制其 Lipschitz 常数——判别器的判断变化不能太剧烈，为生成器保留平滑的梯度信号


## 本章总结

### 降维方法对比

| 方法 | 类型 | 一句话总结 |
|------|------|-----------|
| **PCA (Principal Component Analysis)** | 线性降维 | 见第九章——最大方差方向的线性投影 |
| **t-SNE** | 非线性降维 | PCA 看不到的非线性聚类结构 |
| **UMAP** | 非线性降维 | 比 t-SNE 更快更稳定的替代方案 |

### 生成模型进化

| 方法 | 关键创新 | 局限 |
|------|---------|------|
| **Autoencoder** | 压缩-重建，学习紧凑表示 | 潜在空间不连续，无法采样生成 |
| **VAE** | 概率编码，reparameterization trick | 生成图像偏模糊 |
| **GAN** | 对抗训练，博弈论视角 | 训练不稳定，模式坍塌 |
| **DDcGAN** | 双判别器（全局+局部），条件投影 | 计算开销更高 |

### DDcGAN 要点

| 概念 | 关键设计 |
|------|---------|
| 双判别器 | 全局判别器（整体合理性）+ 局部判别器（细节质量） |
| 条件注入 | 投影判别器 $D(x,c) = w_c^T \phi(x)$ 而非简单拼接 |
| 损失函数 | Hinge Loss 替代交叉熵，训练更稳定 |
| 不对称设计 | 两个判别器结构不同（全局 vs 局部），防止同质化 |
